# 🧩 Notebook: Vector Databases with Chroma

In this notebook we take [Chroma](https://www.trychroma.com/), an open-source vector database, from a single line of Python all the way to a real backend service running in Docker. Vector databases are the piece of infrastructure that makes semantic search, RAG, and "find me things similar to this" possible at scale — instead of comparing a query against every document by hand, they index embeddings so similarity search stays fast as your data grows.

We'll build up gradually: an in-memory client for experimenting, realistic dummy data pulled from Hugging Face `datasets`, a `PersistentClient` that survives a restart, and finally Chroma running as its own container that any service on your machine (or network) can talk to — the shape it actually takes in production.

> **Vector database.** A database that stores *embeddings* (vectors) alongside your original data and lets you search by *similarity* ("what's semantically close to this?") instead of by exact match.

## 📚 Sources

- [Chroma Documentation](https://docs.trychroma.com/)
- [Getting Started](https://docs.trychroma.com/docs/overview/getting-started)
- [Chroma Clients](https://docs.trychroma.com/docs/run-chroma/client)
- [Client-Server Mode](https://docs.trychroma.com/docs/run-chroma/client-server)
- [Manage Collections](https://docs.trychroma.com/docs/collections/manage-collections)
- [Adding Data](https://docs.trychroma.com/docs/collections/add-data) / [Update Data](https://docs.trychroma.com/docs/collections/update-data) / [Delete Data](https://docs.trychroma.com/docs/collections/delete-data)
- [Query and Get](https://docs.trychroma.com/docs/querying-collections/query-and-get)
- [Metadata Filtering](https://docs.trychroma.com/docs/querying-collections/metadata-filtering) / [Full Text Search](https://docs.trychroma.com/docs/querying-collections/full-text-search)
- [Embedding Functions](https://docs.trychroma.com/docs/embeddings/embedding-functions)
- [Configure Collections (HNSW)](https://docs.trychroma.com/docs/collections/configure)
- [Chroma on Docker Hub](https://hub.docker.com/r/chromadb/chroma)
- [Hugging Face `datasets`](https://huggingface.co/docs/datasets/)

Tip: run each code cell with `Shift + Enter`, in order from top to bottom — later cells reuse variables (and running containers!) defined earlier.


## 1. Installing Chroma and your first client

Chroma ships as a single pip package that bundles the client library *and* an embedded server — you don't need anything else running to start experimenting.

```bash
uv add chromadb
```

The simplest possible client, `chromadb.Client()`, is **ephemeral**: it lives entirely in your Python process' memory and disappears the moment the process ends. That's perfect for a first look at the API.


In [1]:
import chromadb

print("chromadb version:", chromadb.__version__)

# An in-memory client - nothing is written to disk, nothing persists between runs.
client = chromadb.Client()

# A "collection" is Chroma's equivalent of a table: a named group of records
# that share an embedding space and can be queried together.
collection = client.create_collection(name="quickstart")

collection.add(
    ids=["1", "2", "3"],
    documents=[
        "The cat sat on the warm windowsill.",
        "Stock markets rallied after the earnings report.",
        "A new rover successfully landed on Mars.",
    ],
)

# .query() does a similarity search: Chroma embeds the query text with the
# same embedding function as the documents, then returns the closest matches.
results = collection.query(query_texts=["space exploration"], n_results=2)
results

chromadb version: 1.5.9


{'ids': [['3', '1']],
 'embeddings': None,
 'documents': [['A new rover successfully landed on Mars.',
   'The cat sat on the warm windowsill.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[1.496218204498291, 1.94489586353302]]}

Notice we never computed an embedding ourselves — Chroma attached a **default embedding function** to the collection automatically and used it both when we called `.add()` and when we called `.query()`. We'll look at embedding functions properly in [section 9](#9-embedding-functions-how-the-text-becomes-vectors).


## 2. Realistic dummy data with Hugging Face `datasets`

Three hand-written sentences are fine for a smoke test, but every real system needs data with some volume and structure to it. Rather than inventing our own, we'll pull a small, public dataset from the [Hugging Face Hub](https://huggingface.co/datasets) via the `datasets` library — the same way you'd pull dummy data for any backend prototype.

We'll use [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news), a classic news-headline classification dataset with four categories (`World`, `Sports`, `Business`, `Sci/Tech`). It's small, loads in a few seconds, and — importantly for this notebook — gives us a clean metadata field (`category`) to filter on later.


In [2]:
from datasets import load_dataset

# Slicing directly in the split string ("train[:300]") avoids downloading and
# materializing the full ~120k-row training split - we only need a sample.
news = load_dataset("fancyzhx/ag_news", split="train[:300]")

category_names = news.features["label"].names  # ['World', 'Sports', 'Business', 'Sci/Tech']
print(category_names)
news[0]

/Users/nils_hellwig/Documents/Seafile/Meine Bibliothek/Lehre/ai-engineering-backend-notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['World', 'Sports', 'Business', 'Sci/Tech']


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [3]:
# Reshape into the flat lists Chroma's .add() expects: parallel arrays of
# ids, documents, and metadatas, all in the same order. `list(...)` matters
# here - dataset columns come back as a `datasets`-specific Column type, and
# Chroma expects plain Python lists.
news_ids = [f"news-{i}" for i in range(len(news))]
news_documents = list(news["text"])
news_metadatas = [{"category": category_names[label]} for label in news["label"]]

print(news_ids[0], "->", news_metadatas[0], "->", news_documents[0])

news-0 -> {'category': 'Business'} -> Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


## 3. `PersistentClient`: surviving a restart

An ephemeral, in-memory client is fine for a notebook, but a real backend needs its data to still be there after a restart. `PersistentClient` writes everything to a folder on disk - still no separate server process, just a different storage backend.


In [4]:
import os

persist_path = os.path.join(os.getcwd(), "content", "chroma_persistent")
os.makedirs(persist_path, exist_ok=True)

persistent_client = chromadb.PersistentClient(path=persist_path)

# get_or_create_collection is idempotent - safe to re-run this cell.
news_collection = persistent_client.get_or_create_collection(name="ag_news")

# Adding an id that already exists is silently ignored (use .upsert() to
# overwrite), so re-running this cell after the first time is also safe.
news_collection.add(ids=news_ids, documents=news_documents, metadatas=news_metadatas)

print("records stored:", news_collection.count())

records stored: 300


That data is now sitting in `content/chroma_persistent/` as SQLite + binary index files. Restart the kernel and re-run just the cell above (skip the `.add()` re-run if you like) and `news_collection.count()` will still report the same number - nothing was recomputed.


## 4. Running Chroma as a backend service (Docker)

A `PersistentClient` still only works from *within the same Python process* that owns the folder on disk. The moment you have more than one service that needs to read or write the same vector data — a web API, a background worker, a second notebook — you need Chroma running as its own standalone process that everyone talks to over the network. That's **client-server mode**, and it's how Chroma actually runs in production.

We'll run the official Chroma image in Docker. It's a single, self-contained container (~180 MB) — no extra dependencies to install on your machine besides Docker itself.

> Make sure Docker Desktop is installed and running first — see [setup.md](../../setup.md) if you haven't done this yet. Everything above this point in the notebook works without Docker; only this section onward needs it.

The cell below pulls the image (pinning a version instead of `:latest`, so the notebook behaves the same way every time you run it), then starts it in the background, publishing port 8000 and mounting a named volume so the data survives a container restart. It's written to be safe to re-run: it first removes any leftover container from a previous run.

(The same image also ships a `chroma run --path <dir>` CLI command if you'd rather run it directly on your machine without Docker at all — installed automatically alongside the `chromadb` pip package. See [Installing the CLI](https://docs.trychroma.com/docs/cli/install) if you want to try that route instead.)

If port 8000 is already taken on your machine, change the left-hand side of `-p 8000:8000` below (e.g. `-p 8080:8000`) and adjust the `port=` argument in the cell after it to match.


In [5]:
# Remove any leftover container from a previous run of this notebook, so
# re-running this cell is always safe.
!docker rm -f chroma-server > /dev/null 2>&1

!docker pull chromadb/chroma:1.5.9
!docker run -d --name chroma-server -p 8000:8000 -v chroma-data:/data chromadb/chroma:1.5.9

import time
time.sleep(3)  # give the container a moment to finish starting up
!docker ps --filter name=chroma-server

1.5.9: Pulling from chromadb/chroma


Digest: sha256:1e0b73a187a28757c572acba508c46f48c9e8b0acaf5c20e6d95cdedce1acdf6
Status: Image is up to date for chromadb/chroma:1.5.9
docker.io/chromadb/chroma:1.5.9



What's next:
    View a summary of image vulnerabilities and recommendations → docker scout quickview chromadb/chroma:1.5.9


61a9f9a48421be42273d24b05fdd6dc912028cc4962c9e33bb3861d5f5a63101


CONTAINER ID   IMAGE                   COMMAND                  CREATED         STATUS         PORTS                                         NAMES
61a9f9a48421   chromadb/chroma:1.5.9   "dumb-init -- chroma…"   3 seconds ago   Up 3 seconds   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp   chroma-server


In [6]:
import chromadb

# HttpClient talks to a Chroma server over HTTP instead of embedding one in
# this process - this is the exact same client interface as before, just
# pointed at a different backend.
http_client = chromadb.HttpClient(host="localhost", port=8000)

print("heartbeat:", http_client.heartbeat())

heartbeat: 1786890043899969507


In [7]:
server_collection = http_client.get_or_create_collection(name="ag_news")
server_collection.add(ids=news_ids, documents=news_documents, metadatas=news_metadatas)

print("records on the server:", server_collection.count())

records on the server: 300


> **In-process vs. client-server, at a glance.**
> - `Client()` / `PersistentClient()`: Chroma runs *inside* your Python process. Simple, zero network overhead, but only that one process can use it.
> - `HttpClient()`: Chroma runs as its *own* process (in our case, in a Docker container). Any number of clients — different notebooks, a FastAPI backend, a scheduled job — can connect to it concurrently. This is the shape a real deployment takes.

From here on, we'll keep using `server_collection` (backed by the Docker container) so everything we do reflects how you'd actually operate Chroma as a backend.


## 5. Collections in depth

A collection is more than just a bucket of records — it has a name, optional metadata, and an index configuration. A few rules and operations worth knowing:

- **Naming rules:** 3–512 characters, must start and end with a lowercase letter or digit, no consecutive dots, and can't look like an IP address.
- **`create_collection`** fails if the name already exists; **`get_or_create_collection`** is idempotent (creates it if missing, returns the existing one otherwise) — usually what you want in application code.
- **`list_collections`** enumerates what's on a client, with `limit`/`offset` for pagination.
- **`modify()`** renames a collection or updates its metadata in place.
- **`count()`** / **`peek()`** are cheap ways to sanity-check a collection without running a full query.
- **`delete_collection`** is destructive and irreversible — it drops the collection and everything in it.


In [8]:
print([c.name for c in http_client.list_collections()])

# Collection-level metadata is just a dict you attach at creation time -
# handy for things like "which embedding model generated this" or a
# human-readable description.
demo_collection = http_client.get_or_create_collection(
    name="demo_collection",
    metadata={"description": "scratch collection for this notebook's examples"},
)

demo_collection.modify(metadata={"description": "renamed via .modify()"})
print(demo_collection.metadata)

print("count:", server_collection.count())
server_collection.peek(limit=2)

['ag_news']
{'description': 'renamed via .modify()'}
count: 300


{'ids': ['news-0', 'news-1'],
 'embeddings': array([[ 7.43857860e-03,  2.85623740e-02,  4.10954100e-02,
          1.05001400e-01,  2.32820730e-02,  3.51258400e-02,
         -2.14060540e-02, -2.29022880e-02,  4.94526300e-03,
         -6.68986400e-02, -5.93040550e-02,  2.45947970e-02,
         -5.12895620e-02, -4.04271930e-02,  6.65476400e-04,
         -1.17221430e-02, -4.34829150e-03,  2.06902000e-02,
         -2.29348780e-03, -3.47105500e-02, -8.01126660e-02,
         -6.02049900e-02, -2.86628940e-02, -1.86906400e-02,
         -7.84591660e-02,  4.60197140e-02, -4.96720860e-02,
         -3.33419900e-02,  5.32347600e-02, -1.26492070e-01,
         -7.07766400e-02, -6.30797900e-04, -2.26125980e-02,
          2.00373700e-03,  8.22305400e-03, -6.30252000e-03,
          8.53443600e-03, -4.64320600e-02,  4.35815700e-02,
          3.90969030e-02,  1.36761360e-03, -2.62945830e-02,
         -5.70246880e-02,  3.29649370e-02,  1.93821500e-02,
          4.52636630e-02,  5.92519340e-02,  5.84027000e-

In [9]:
# Clean up the scratch collection - delete_collection is irreversible.
http_client.delete_collection(name="demo_collection")
print([c.name for c in http_client.list_collections()])

['ag_news']


## 6. Adding, updating, upserting, and deleting data

Four operations cover everything you'll do to a collection's contents:

- **`.add()`** — insert new records. If you pass `documents` (and no precomputed `embeddings`), Chroma embeds them automatically. Adding an `id` that already exists is **silently ignored** — `.add()` never overwrites.
- **`.update()`** — change fields on existing records by `id`. Unknown ids are logged and ignored. Passing new `documents` without new `embeddings` re-embeds them.
- **`.upsert()`** — update if the id exists, insert if it doesn't. The one you reach for most often in practice.
- **`.delete()`** — remove records, either by explicit `ids=[...]` or, more powerfully, with a `where` metadata filter that deletes everything matching it.


In [10]:
crud = http_client.get_or_create_collection(name="crud_demo")

crud.add(
    ids=["a", "b"],
    documents=["first draft of document A", "first draft of document B"],
    metadatas=[{"status": "draft"}, {"status": "draft"}],
)

# This id already exists, so .add() quietly does nothing for it - "a" keeps
# its original text. Use .update() or .upsert() to actually change it.
crud.add(ids=["a"], documents=["this text will NOT be stored"])
print("still the original:", crud.get(ids=["a"])["documents"])

# .update() DOES change it, and re-embeds since we gave new document text.
crud.update(ids=["a"], documents=["revised document A"], metadatas=[{"status": "reviewed"}])
print("after update:", crud.get(ids=["a"]))

# .upsert() inserts "c" (new) and updates "b" (existing) in a single call.
crud.upsert(
    ids=["b", "c"],
    documents=["revised document B", "brand new document C"],
    metadatas=[{"status": "reviewed"}, {"status": "draft"}],
)
print("all records:", crud.get())

still the original: ['first draft of document A']
after update: {'ids': ['a'], 'embeddings': None, 'metadatas': [{'status': 'reviewed'}], 'documents': ['revised document A'], 'data': None, 'uris': None, 'included': ['metadatas', 'documents']}


all records: {'ids': ['a', 'b', 'c'], 'embeddings': None, 'metadatas': [{'status': 'reviewed'}, {'status': 'reviewed'}, {'status': 'draft'}], 'documents': ['revised document A', 'revised document B', 'brand new document C'], 'data': None, 'uris': None, 'included': ['metadatas', 'documents']}


In [11]:
# Delete by explicit id...
crud.delete(ids=["c"])

# ...or by a metadata filter, which can remove many records in one call.
crud.delete(where={"status": "draft"})

print("remaining:", crud.get())
http_client.delete_collection(name="crud_demo")

remaining: {'ids': ['a', 'b'], 'embeddings': None, 'metadatas': [{'status': 'reviewed'}, {'status': 'reviewed'}], 'documents': ['revised document A', 'revised document B'], 'data': None, 'uris': None, 'included': ['metadatas', 'documents']}


## 7. Querying: `.query()` vs. `.get()`

Chroma gives you two different ways to read data back, and picking the right one matters:

- **`.query(query_texts=..., n_results=...)`** — a *similarity search*. Embeds the query, finds the `n_results` nearest neighbors by vector distance. This is "search" in the sense of RAG/semantic search.
- **`.get(...)`** — a *plain lookup/filter*, no embeddings or distances involved. Use it when you know exactly what you want by `id` or metadata, with `limit`/`offset` for pagination — closer to a normal database `SELECT ... WHERE`.

The `include` parameter controls which fields come back (`documents`, `metadatas`, `distances`, `embeddings` — `ids` are always included). Leaving embeddings out by default keeps result payloads small.


In [12]:
# Similarity search: "closest in meaning", not "contains these words".
similar = server_collection.query(
    query_texts=["a company's shares went up"],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)
for doc, meta, dist in zip(similar["documents"][0], similar["metadatas"][0], similar["distances"][0]):
    print(f"[{dist:.3f}] ({meta['category']}) {doc[:70]}")

[0.994] (Business) Stocks End Up, But Near Year Lows (Reuters) Reuters - Stocks ended sli
[1.099] (Business) HP shares tumble on profit news Hewlett-Packard shares fall after disa
[1.282] (Business) Google IPO: Type in 'confusing,' 'secrecy' I've submitted my bid to bu


In [13]:
# Plain filtered lookup - no ranking, no embeddings involved, just "give me
# up to 5 Sci/Tech articles".
sci_tech = server_collection.get(where={"category": "Sci/Tech"}, limit=5)
for doc in sci_tech["documents"]:
    print("-", doc[:70])

- 'Madden,' 'ESPN' Football Score in Different Ways (Reuters) Reuters - 
- Group to Propose New High-Speed Wireless Format (Reuters) Reuters - A 
- AOL to Sell Cheap PCs to Minorities and Seniors (Reuters) Reuters - Am
- Companies Approve New High-Capacity Disc Format (Reuters) Reuters - A 
- Missing June Deals Slow to Return for Software Cos. (Reuters) Reuters 


## 8. Metadata filtering and full-text search

`.query()` and `.get()` both accept a `where` filter over metadata fields, and `.get()`/`.query()` can additionally take a `where_document` filter over the document text itself.

**`where` operators:** `$eq` (default if you just pass a plain value), `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, combined with `$and`/`$or`. Array-valued metadata additionally supports `$contains`/`$not_contains`.

**`where_document` operators:** `$contains`/`$not_contains` (case-sensitive substring match) and `$regex`/`$not_regex` for pattern matching.


In [14]:
# $in: category is one of several values
some_categories = server_collection.get(
    where={"category": {"$in": ["Sports", "Business"]}},
    limit=4,
)
print([m["category"] for m in some_categories["metadatas"]])

# $and: combine two conditions
world_or_business = server_collection.query(
    query_texts=["international relations"],
    n_results=3,
    where={"$or": [{"category": "World"}, {"category": "Business"}]},
)
[m["category"] for m in world_or_business["metadatas"][0]]

['Business', 'Business', 'Business', 'Business']


['Business', 'Business', 'Business']

In [15]:
# where_document: filter by the text itself, independent of metadata.
mentions_president = server_collection.get(
    where_document={"$contains": "president"},
    limit=3,
)
print(len(mentions_president["ids"]), "documents mention 'president'")

# Combine a metadata filter with a full-text filter in one call.
world_news_about_president = server_collection.get(
    where={"category": "World"},
    where_document={"$contains": "president"},
    limit=3,
)
for doc in world_news_about_president["documents"]:
    print("-", doc[:80])

3 documents mention 'president'


## 9. Embedding functions: how the text becomes vectors

Every collection has an **embedding function** attached to it, invoked automatically on `.add()`, `.update()`, `.upsert()`, and `.query()` whenever you pass raw text instead of precomputed vectors. Unless you specify one, Chroma uses `DefaultEmbeddingFunction` — a small [Sentence Transformers](https://www.sbert.net/) model (`all-MiniLM-L6-v2`, 384 dimensions) that runs **locally**, with no API key and no network call.


In [16]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# You can call an embedding function directly - handy for debugging, or for
# precomputing embeddings yourself before calling .add(embeddings=...).
default_ef = DefaultEmbeddingFunction()
vectors = default_ef(["a short sentence", "another one"])

print("number of vectors:", len(vectors))
print("dimensions per vector:", len(vectors[0]))

number of vectors: 2
dimensions per vector: 384


For production use cases you'll often want a hosted, higher-quality embedding model instead. Chroma has built-in embedding functions for most major providers (OpenAI, Cohere, Google Gemini, Voyage AI, Hugging Face, Ollama, and more) — you attach one at collection-creation time, and it's used consistently for every add/update/query afterwards:

```python
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

openai_ef = OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",  # reads the key from the environment, not hard-coded
    model_name="text-embedding-3-small",
)

collection = client.create_collection(name="my_collection", embedding_function=openai_ef)
```

The embedding function's config is saved as part of the collection, so re-opening it later (even from a different process) automatically uses the right one. If none of the built-in options fit, Chroma also lets you write a fully custom `EmbeddingFunction` — see [Embedding Functions](https://docs.trychroma.com/docs/embeddings/embedding-functions) for the interface.


## 10. Collection tuning basics (HNSW)

### The problem this solves

When you call `.query()`, Chroma has to find the handful of vectors that are closest to your query vector out of everything in the collection. The naive way to do that is to compare your query against *every single vector* and rank them — this is called a **brute-force** or **exact** search. It gives you a perfectly correct answer, but it gets slower in direct proportion to how much data you have: 10x the documents means (roughly) 10x the time per query. That doesn't scale to millions of vectors.

So instead, Chroma (like virtually every vector database) uses an **approximate nearest neighbor (ANN)** algorithm called **HNSW** (Hierarchical Navigable Small World). "Approximate" is the key word: it deliberately does *not* check every vector. Instead, at insert time it builds a graph that connects each vector to a handful of others that are close to it — similar to the "six degrees of separation" idea in a social network, where you can hop from any person to any other person through surprisingly few connections. To answer a query, HNSW starts somewhere in the graph and repeatedly hops to whichever neighbor gets it closer to the query, instead of visiting every single node. That's what makes it fast: query time grows very slowly as your collection grows, instead of linearly.

The price for that speed is that HNSW might occasionally miss the *true* single best match in favor of a very-close second-best — because it only explored a subset of the graph, not the whole thing. How often that happens, and how fast it happens, is what the settings below control. **For the collection sizes you'll work with in this notebook (hundreds to low thousands of records), the default settings are already both fast and accurate enough — you won't need to touch any of this.** It becomes relevant once a collection grows into the hundreds of thousands or millions of vectors.

### The three settings

- **`space`** — *which* distance metric to use when comparing two vectors: `cosine` (angle between vectors — the most common choice for text embeddings), `l2` (straight-line/Euclidean distance), or `ip` (inner product). This isn't a speed/accuracy tradeoff, it's a correctness one: use whatever metric your embedding model was trained/designed for. For the sentence-embedding models used in this notebook, that's `cosine`.
- **`ef_search`** — *how thoroughly* to explore the graph when answering a query, i.e. how many candidate neighbors to look at before returning the top results. Think of it as "how many extra side-streets do I check before committing to the fastest-looking route". A small value (e.g. `10`) explores very little: quick, but more likely to miss the best answer. A large value (e.g. `100`) explores much more: slower per query, but much more likely to find the true best matches. This is the main dial you'd turn if you notice search results feel like they're missing the answer you expected — and, unlike the setting below, it can be changed on an existing collection at any time via `.modify()`.
- **`ef_construction`** — the same idea, but applied *once*, when the graph is first being built (i.e. every time you `.add()` data). A higher value builds a better-connected, higher-quality graph — which then makes *every future query* more accurate for the same `ef_search` — at the cost of `.add()` calls taking longer.

**Rule of thumb:** `ef_search` trades query speed for query accuracy, every time you search. `ef_construction` trades indexing speed for the *quality of the graph* those searches run against, once, when you add data. Concretely: on a 50,000-vector collection with 2048-dimensional embeddings, going from `ef_search=10` to `ef_search=100` (combined with a larger `ef_construction`) can noticeably improve how often the "true" best matches actually show up in your results — but each query also does more work to get there. Only turn these dials once you've actually measured a recall problem (e.g. results that feel obviously worse than they should be) — don't tune blind.


In [17]:
tuned_collection = http_client.create_collection(
    name="tuned_demo",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 200,  # higher = better index quality, slower to build
            "ef_search": 50,         # higher = better recall, slower per query
        }
    },
)
print(tuned_collection.configuration_json["hnsw"])
http_client.delete_collection(name="tuned_demo")

{'space': 'cosine', 'ef_construction': 200, 'ef_search': 50, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}


## 11. Conditional transactions (preview)

For read-check-write workflows — "read a record, decide what to do based on its current value, write the result, but only if nothing else changed it in the meantime" — Chroma's docs describe **conditional transactions**: optimistic, collection-scoped transactions where conflicts are detected at commit time rather than locking anything upfront.

```python
txn = collection.conditional()

existing = txn.get(ids=["doc-1"])
if existing["ids"]:
    txn.update(ids=["doc-1"], documents=["updated document"])

txn.commit()

# Or, to auto-retry on a conflict instead of handling it yourself:
def update_doc(txn):
    existing = txn.get(ids=["doc-1"])
    if existing["ids"]:
        txn.update(ids=["doc-1"], documents=["updated document"])

collection.conditional().run(update_doc, max_retries=3)
```

**Limitations to know about:** a transaction is scoped to a single collection, there's no nesting, `txn.query()` isn't supported (only `get()`), deletes inside a transaction need explicit ids (no `where` filters), and each transaction can buffer at most one write per id.

> This feature was documented on `docs.trychroma.com` but `collection.conditional()` isn't available yet in `chromadb==1.5.9` (the version this notebook pins) — it raises `AttributeError`. Treat the snippet above as a preview of where the API is heading, check the [Conditional Transactions docs](https://docs.trychroma.com/docs/collections/conditional-transactions) for current availability, and reach for a plain `.get()` → check → `.update()` in your own code until then (accepting that, without the transaction, a concurrent writer could race you).


## 12. Wrap-up

We went from a three-line in-memory client to Chroma running as an independent backend service in Docker, talked to over HTTP — the same shape it takes in a real deployment. Along the way: loading realistic dummy data from Hugging Face, the full data lifecycle (`add`/`update`/`upsert`/`delete`), the difference between similarity search and plain filtering, combining metadata and full-text filters, where embeddings actually come from, and the tuning knobs available once a collection gets large.

When you're eventually done experimenting (after the exercises below, since they still use the running server), either pause the container (`docker stop chroma-server` — its data stays safe in the `chroma-data` volume, and `docker start chroma-server` brings it back exactly as you left it), or fully tear it down with the cleanup cell at the very end of this notebook.

**Next up:** more backend chapters — serving, deployment, CI/CD for AI applications — will build on this one.


## 13. Exercises

### Exercise 1: Semantic search with a metadata filter

**Task:** Using `server_collection` (the Docker-backed AG News collection), write a query that finds the 3 articles most similar to `"a major sports victory"`, but restricted to the `"Sports"` category using a `where` filter. Print each result's distance and first 80 characters.


In [18]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
results = server_collection.query(
    query_texts=["a major sports victory"],
    n_results=3,
    where={"category": "Sports"},
    include=["documents", "distances"],
)

for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc[:80]}")
```

</details>


### Exercise 2: Fix the broken filter

**Task:** The query below is meant to find articles that are in either the `"World"` or `"Sci/Tech"` category *and* whose text contains the word `"China"`. It raises an error. Find the bug and fix it — the `where`/`where_document` operator syntax is the culprit.

```python
broken = server_collection.get(
    where={"category": ["World", "Sci/Tech"]},
    where_document={"contains": "China"},
)
```


In [19]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

Two bugs: `where` needs the `$in` operator to match against a list of values (a bare list isn't valid), and `where_document` needs the operator key spelled `$contains`, not `contains`.

```python
fixed = server_collection.get(
    where={"category": {"$in": ["World", "Sci/Tech"]}},
    where_document={"$contains": "China"},
)
print(len(fixed["ids"]), "matches")
```

</details>


### Exercise 3: From ephemeral to durable

**Task:** Create a brand-new in-memory client and collection, add two documents of your choice to it, then migrate that data into a *new* collection on the Docker server (`http_client`) using `.get()` to read it back out and `.add()` to write it into the new one. Confirm the server-side collection has the right count afterwards. Clean up by deleting the collection you created on the server.


In [20]:
# Your solution here:


<details>
<summary><b>Show Solution</b></summary>

```python
scratch_client = chromadb.Client()
scratch_collection = scratch_client.create_collection(name="scratch")
scratch_collection.add(
    ids=["x", "y"],
    documents=["a document only I have right now", "a second one"],
)

exported = scratch_collection.get(include=["documents", "metadatas"])

migrated = http_client.create_collection(name="migrated_demo")
migrated.add(
    ids=exported["ids"],
    documents=exported["documents"],
    metadatas=exported["metadatas"],
)

print("migrated count:", migrated.count())

http_client.delete_collection(name="migrated_demo")
```

</details>


## 14. Cleanup

Now that the exercises are done, tear down the Docker container and its volume so nothing from this notebook is left running on your machine:


In [21]:
# Full teardown: removes the container and its named volume, so nothing
# from this notebook's Docker section is left behind on your machine.
!docker rm -f chroma-server
!docker volume rm chroma-data

chroma-server


chroma-data
